# Advanced DOT Traffic Analysis 
This pipeline ingests NYC Yellow Taxi data, maps it to real neighborhoods, categorizes trips into Rush Hour windows, and joins an external Weather API to prove how rain impacts congestion.

In [6]:
import pandas as pd
import numpy as np
import requests

# --- 1. INGEST MULTIPLE SOURCES ---
print("Loading Parquet and CSV data...")
df_trips = pd.read_parquet('data/yellow_tripdata_2026-04.parquet')
df_zones = pd.read_csv('data/taxi_zone_lookup.csv')

print("Fetching live historical weather data from Open-Meteo API...")
url = "https://archive-api.open-meteo.com/v1/archive?latitude=40.7143&longitude=-74.006&start_date=2026-04-01&end_date=2026-04-30&hourly=precipitation"
weather_data = requests.get(url).json()

# Convert weather JSON to a Pandas DataFrame
df_weather = pd.DataFrame(weather_data['hourly'])
df_weather['time'] = pd.to_datetime(df_weather['time'])
df_weather['is_raining'] = df_weather['precipitation'] > 0

print(f"Successfully loaded {len(df_trips):,} trips, {len(df_zones)} zones, and {len(df_weather)} hours of weather data.")

Loading Parquet and CSV data...
Fetching live historical weather data from Open-Meteo API...
Successfully loaded 3,831,240 trips, 265 zones, and 720 hours of weather data.


In [7]:
# --- 2. VALIDATE & CLEAN DATA ---
initial_count = len(df_trips)

# Calculate duration in minutes
df_trips['trip_duration_minutes'] = (df_trips['tpep_dropoff_datetime'] - df_trips['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# Apply rules (No teleportation, no 0 distances)
df_clean = df_trips[
    (df_trips['trip_distance'] > 0) & 
    (df_trips['trip_duration_minutes'] > 0) & 
    (df_trips['trip_duration_minutes'] < 300)
].copy()

df_clean['speed_mph'] = df_clean['trip_distance'] / (df_clean['trip_duration_minutes'] / 60.0)
df_clean = df_clean[df_clean['speed_mph'] <= 80]

print(f"Dropped {initial_count - len(df_clean):,} invalid rows.")

Dropped 144,220 invalid rows.


In [8]:
# --- 3. MODELING TEMPORAL & WEATHER WORKFLOW ---
print("Applying temporal and weather logic...")

# Map Zone Names
df_clean = df_clean.merge(df_zones[['LocationID', 'Zone']], left_on='PULocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'Zone': 'Pickup_Zone'})
df_clean = df_clean.merge(df_zones[['LocationID', 'Zone']], left_on='DOLocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'Zone': 'Dropoff_Zone'})
df_clean['Route'] = df_clean['Pickup_Zone'] + " to " + df_clean['Dropoff_Zone']

# Temporal Modeling (Rush Hour)
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour
def categorize_time(hour):
    if 7 <= hour <= 9: return 'Morning Rush (7-9AM)'
    elif 16 <= hour <= 19: return 'Evening Rush (4-7PM)'
    else: return 'Off-Peak'
df_clean['time_of_day'] = df_clean['pickup_hour'].apply(categorize_time)

# Weather Merge (Round trip time down to nearest hour to match weather data)
df_clean['weather_join_time'] = df_clean['tpep_pickup_datetime'].dt.floor('H')
df_clean = df_clean.merge(df_weather[['time', 'is_raining']], left_on='weather_join_time', right_on='time', how='left')

df_clean[['tpep_pickup_datetime', 'time_of_day', 'is_raining', 'speed_mph']].head()

Applying temporal and weather logic...


/var/folders/6n/fzsly8555t9f4pb8gyyxyb8r0000gn/T/ipykernel_27228/1088319001.py:20: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_clean['weather_join_time'] = df_clean['tpep_pickup_datetime'].dt.floor('H')


,tpep_pickup_datetime,time_of_day,is_raining,speed_mph
0,2026-04-01 00:40:05,Off-Peak,False,13.280632
1,2026-04-01 00:09:19,Off-Peak,False,36.345205
2,2026-04-01 00:15:29,Off-Peak,False,24.512000
3,2026-04-01 00:14:20,Off-Peak,False,35.154512
4,2026-04-01 00:04:53,Off-Peak,False,11.458432


### Final Output: Proving that Rain + Rush Hour ruins NYC Traffic

In [9]:
# --- 4. ADVANCED METRICS OUTPUT ---
# Let's compare speed by Rush Hour and Rain!
advanced_stats = df_clean.groupby(['time_of_day', 'is_raining']).agg(
    average_speed_mph=('speed_mph', 'mean'),
    total_trips=('speed_mph', 'count')
).reset_index()

print("✅ Pipeline complete! Check out the impact of Rain and Rush Hour on City-Wide Speed:")
display(advanced_stats.sort_values(by='average_speed_mph'))

# Save it for your presentation
advanced_stats.to_csv('advanced_traffic_report.csv', index=False)

✅ Pipeline complete! Check out the impact of Rain and Rush Hour on City-Wide Speed:


,time_of_day,is_raining,average_speed_mph,total_trips
0,Evening Rush (4-7PM),False,9.405898,760903
1,Evening Rush (4-7PM),True,9.587689,138350
3,Morning Rush (7-9AM),True,10.825825,56430
2,Morning Rush (7-9AM),False,10.964192,377552
4,Off-Peak,False,11.085243,2087293
5,Off-Peak,True,11.194823,266482
